#### Análsis Dataset Limpio

In [2]:
import pandas as pd 
import os


ruta_archivo = os.path.join('..', 'datos', 'processed', '01_incendios_CV_limpio.csv')
df_revisado = pd.read_csv(ruta_archivo, sep=';')


print("--- PRIMERAS 5 FILAS DEL DATASET LIMPIO ---")
print(df_revisado.head())

print("\n--- RESUMEN TÉCNICO ---")
print(df_revisado.info())


print("\n--- ESTADÍSTICAS RÁPIDAS ---")
print(df_revisado[['duracion_h', 'Superficie_Total_Real', 'lat', 'lon']].describe())

--- PRIMERAS 5 FILAS DEL DATASET LIMPIO ---
   Campania            fecha_ini  duracion_h Provincia             Municipio  \
0      2010  2010-01-01 14:48:00   23.200000  ALICANTE                AIGÜES   
1      2010  2010-02-10 13:50:00    0.916667  ALICANTE  GUARDAMAR DEL SEGURA   
2      2010  2010-03-02 13:35:00    4.750000  ALICANTE              BENIFATO   
3      2010  2010-03-09 12:30:00    1.166667  ALICANTE              BENIDORM   
4      2010  2010-03-15 16:15:00    2.750000  ALICANTE       MONÒVER/MONÓVAR   

                                              Causa  Superficie_Total_Real  \
0                               [400]  Intencionado                  15.00   
1                               [400]  Intencionado                   0.01   
2       [284]  Limpieza de lindes y bordes de finca                   0.75   
3                               [400]  Intencionado                   0.02   
4  [399]  Otras causas no intencionales (conocidas)                   0.40   

   Sup

#### Retoques Dataset Final

In [3]:
import pandas as pd


df_revisado['fecha_ini'] = pd.to_datetime(df_revisado['fecha_ini'])


df_revisado['Causa'] = df_revisado['Causa'].str.replace(r'\[.*\]\s*', '', regex=True)


df_revisado = df_revisado[
    (df_revisado['lon'] > -1.7) & (df_revisado['lon'] < 0.7) &
    (df_revisado['lat'] > 37.5) & (df_revisado['lat'] < 41.0)
]


print(f"Registros tras filtrar errores geográficos: {len(df_revisado)}")
print("\nPrimeras causas limpias:")
print(df_revisado['Causa'].unique()[:5]) 


df_revisado['lat'] = df_revisado['lat'].round(6)
df_revisado['lon'] = df_revisado['lon'].round(6) 
df_revisado['lat'] = pd.to_numeric(df_revisado['lat'], errors='coerce')
df_revisado['lon'] = pd.to_numeric(df_revisado['lon'], errors='coerce')



df_revisado.to_csv(ruta_archivo, sep=';', index=False, encoding='utf-8-sig')

Registros tras filtrar errores geográficos: 4477

Primeras causas limpias:
['Intencionado' 'Limpieza de lindes y bordes de finca'
 'Otras causas no intencionales (conocidas)'
 'Quema agrícola (quema de rastrojos)' 'Otras quemas de limpieza']


#### Revisión Dataset Final

In [6]:
import pandas as pd
import os


ruta_archivo = os.path.join('..', 'datos', 'processed', '01_incendios_CV_limpio.csv')
df = pd.read_csv(ruta_archivo, sep=';')

print("---  INFORME DE AUDITORÍA DEL DATASET ---")


fuera_lat = df[(df['lat'] < 37.8) | (df['lat'] > 40.8)]
fuera_lon = df[(df['lon'] < -1.6) | (df['lon'] > 0.6)]
print(f"\n Geografía:")
print(f"   - Puntos fuera de rango Latitud: {len(fuera_lat)}")
print(f"   - Puntos fuera de rango Longitud: {len(fuera_lon)}")


errores_superficie = df[df['Superficie_Total_Real'] < df['Superficie_Arbolada']]
print(f"\n Superficies:")
print(f"   - Errores de lógica (Total < Arbolada): {len(errores_superficie)}")
print(f"   - Incendio más grande: {df['Superficie_Total_Real'].max()} ha")


print(f"\n Muestra de Causas (deben estar limpias de códigos):")
print(df['Causa'].unique()[:5])


print(f"\n Tipos de Datos Reales:")
print(df.dtypes)


print("\n TOP 3 INCENDIOS MÁS GRANDES (Verificación de impacto):")
print(df.nlargest(3, 'Superficie_Total_Real')[['fecha_ini', 'Municipio', 'Superficie_Total_Real', 'lat', 'lon']])

--- 🛡️ INFORME DE AUDITORÍA DEL DATASET ---

📍 Geografía:
   - Puntos fuera de rango Latitud: 0
   - Puntos fuera de rango Longitud: 0

🌳 Superficies:
   - Errores de lógica (Total < Arbolada): 0
   - Incendio más grande: 30691.39 ha

🔥 Muestra de Causas (deben estar limpias de códigos):
['Intencionado' 'Limpieza de lindes y bordes de finca'
 'Otras causas no intencionales (conocidas)'
 'Quema agrícola (quema de rastrojos)' 'Otras quemas de limpieza']

📊 Tipos de Datos Reales:
Campania                               int64
fecha_ini                             object
duracion_h                           float64
Provincia                             object
Municipio                             object
Causa                                 object
Superficie_Total_Real                float64
Superficie_Arbolada                  float64
lat                                  float64
lon                                  float64
AfectoZonasInterfazUrbanoForestal     object
AfectoEspacioProtegido 